In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Set model to evaluation mode
model.eval()

def get_bert_embedding(text):
    
    # Tokenize and prepare input
    inputs = tokenizer(text, return_tensors='pt', truncation=True, 
                      max_length=512, padding=True)
    
    # Get embeddings without gradient calculation
    with torch.no_grad():
        outputs = model(**inputs)
    
   
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    
    return embedding

def calculate_cosine_similarity_bert(df, ref_col='reference_answer', 
                                     sub_col='submitted_answer'):
    
    similarities = []
    
    for idx, row in df.iterrows():
        ref_text = str(row[ref_col])
        sub_text = str(row[sub_col])
        
        # Get BERT embeddings
        ref_embedding = get_bert_embedding(ref_text)
        sub_embedding = get_bert_embedding(sub_text)
        
        # Calculate cosine similarity
        similarity = cosine_similarity(
            ref_embedding.reshape(1, -1), 
            sub_embedding.reshape(1, -1)
        )[0][0]
        
        similarities.append(similarity)
        
        if (idx + 1) % 10 == 0:
            print(f"Processed {idx + 1} rows...")
    
    df['cosine_similarity'] = similarities
    return df

# Calculate cosine similarity for both datasets
print("Processing benchmark_df...")
benchmark_df = calculate_cosine_similarity_bert(benchmark_df)

print("\nProcessing collected_df...")
collected_df = calculate_cosine_similarity_bert(collected_df)

# Display results
print("\nBenchmark Dataset - Sample Results:")
print(benchmark_df[['reference_answer', 'submitted_answer', 'cosine_similarity']].head())

print("\nCollected Dataset - Sample Results:")
print(collected_df[['reference_answer', 'submitted_answer', 'cosine_similarity']].head())

# Summary statistics
print("\nBenchmark Dataset - Similarity Statistics:")
print(benchmark_df['cosine_similarity'].describe())

print("\nCollected Dataset - Similarity Statistics:")
print(collected_df['cosine_similarity'].describe())